# RAG sobre Historia de las Guerras Mundiales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhonattanreales21/nlp_icesi/blob/main/unidad_6/Cano_Reales_RAG.ipynb)

**Autores:** Andrés Cano & Jhonattan Reales  
**Curso:** Procesamiento de Lenguaje Natural — Maestría MIAA, ICESI  
**Unidad 6:** Retrieval-Augmented Generation (RAG)

---

## Objetivo

En este taller construimos un **chatbot conversacional basado en RAG** especializado en la historia de las Guerras Mundiales. A diferencia de un LLM estándar, nuestro sistema recupera fragmentos relevantes de una base de conocimiento propia (artículos de Wikipedia en inglés) antes de generar cada respuesta, lo que garantiza mayor precisión y trazabilidad de las fuentes.

## Componentes del sistema

| Componente | Elección | Justificación |
|---|---|---|
| Dataset | `wikimedia/wikipedia` (filtrado WW1/WW2) | Corpus histórico verificable, en inglés |
| LLM | `mistral:7b-instruct` (Ollama) | 7B params, mejor razonamiento que llama3.2:3b |
| Embeddings | `intfloat/multilingual-e5-large` | Multilingüe, alto rendimiento en inglés |
| Vector Store | FAISS | Indexación vectorial eficiente y local |
| Framework | LangChain + Gradio | Abstracción de cadenas + interfaz conversacional |

In [ ]:
import pkg_resources
import warnings

warnings.filterwarnings("ignore")

# Detectamos si estamos en Google Colab para instalar dependencias sólo si es necesario
installed_packages = [pkg.key for pkg in pkg_resources.working_set]
IN_COLAB = "google-colab" in installed_packages
print(f"Ejecutando en Colab: {IN_COLAB}")

In [ ]:
# Instalamos todas las dependencias necesarias (sólo en Colab)
# faiss-gpu-cu12 aprovecha la GPU T4 de Colab para búsqueda vectorial acelerada
if IN_COLAB:
    !pip install -q \
        langchain langchain-ollama langchain-community \
        langchain-huggingface langchain-text-splitters \
        faiss-gpu-cu12 sentence-transformers \
        datasets gradio wordcloud \
        matplotlib seaborn nltk colab-xterm

---
## Introducción: ¿Qué es RAG?

**Retrieval-Augmented Generation (RAG)** es una arquitectura que combina dos etapas:

```
Pregunta del usuario
       │
       ▼
┌─────────────────┐     ┌──────────────────────┐
│  RETRIEVE       │────▶│  Vector Store (FAISS) │
│  Buscar chunks  │◀────│  Artículos Wikipedia  │
│  relevantes     │     └──────────────────────┘
└────────┬────────┘
         │ contexto recuperado
         ▼
┌─────────────────┐
│  AUGMENT        │  Inyectar contexto en el prompt
└────────┬────────┘
         ▼
┌─────────────────┐
│  GENERATE       │  LLM genera respuesta fundamentada
│  Mistral 7B     │
└─────────────────┘
```

### ¿Por qué RAG en lugar de un LLM estándar?

1. **Alucinaciones reducidas**: el LLM sólo puede responder con lo que está en el corpus recuperado.
2. **Trazabilidad**: cada respuesta incluye las fuentes (artículos de Wikipedia) que la fundamentan.
3. **Conocimiento actualizable**: basta con re-indexar documentos nuevos sin reentrenar el modelo.
4. **Eficiencia**: usar un modelo pequeño (7B) con un buen retriever supera a un modelo grande sin contexto.

---
## 1. Carga del Dataset

Usamos **`wikimedia/wikipedia`** (dump 20231101 en inglés) cargado en modo **streaming** para evitar descargar el dataset completo (~20 GB). Filtramos artículos cuyos títulos contengan palabras clave relacionadas con la Primera y Segunda Guerra Mundial.

### Palabras clave de filtrado

Cubrimos eventos, lugares, organizaciones y tecnologías representativas de ambas guerras para obtener un corpus temáticamente rico y equilibrado.

In [ ]:
from datasets import load_dataset
import pandas as pd

# Palabras clave que definen nuestro corpus de Guerras Mundiales
WW_KEYWORDS = [
    # General
    "World War",
    # WWI específico
    "Western Front",
    "Eastern Front",
    "Trench warfare",
    "Armistice",
    "Treaty of Versailles",
    "Gallipoli",
    "Somme",
    "Verdun",
    "Austro-Hungarian",
    "Ottoman Empire",
    # WWII específico
    "Blitzkrieg",
    "Holocaust",
    "D-Day",
    "Normandy",
    "Nazi",
    "Wehrmacht",
    "Luftwaffe",
    "RAF",
    "Pearl Harbor",
    "Hiroshima",
    "Stalingrad",
    "El Alamein",
    "Operation Overlord",
    "Battle of Berlin",
    # Batallas (ambas guerras)
    "Battle of the",
]

MAX_ARTICLES = 3000  # límite para mantener tiempos de embedding razonables

print("Cargando dataset Wikipedia en modo streaming...")
wiki_stream = load_dataset(
    "wikimedia/wikipedia", "20231101.en", split="train", streaming=True
)

articles = []
for example in wiki_stream:
    title = example["title"]
    # Filtramos por keywords en el título (case-insensitive)
    if any(kw.lower() in title.lower() for kw in WW_KEYWORDS):
        articles.append(
            {
                "title": title,
                "url": example["url"],
                "text": title + "\n\n" + example["text"],  # título + cuerpo
            }
        )
    if len(articles) >= MAX_ARTICLES:
        break

print(f"Artículos recuperados: {len(articles)}")
print("Muestra de títulos:")
for a in articles[:10]:
    print(f'  - {a["title"]}')

In [ ]:
# Convertimos a DataFrame para facilitar la exploración
pd.set_option("display.max_colwidth", 120)
df = pd.DataFrame(articles)

# Calculamos longitud en palabras de cada artículo
df["word_count"] = df["text"].apply(lambda t: len(t.split()))
df["char_count"] = df["text"].apply(len)

print(f"Shape del DataFrame: {df.shape}")
df[["title", "word_count", "char_count"]].head(10)

---
## 2. Análisis Exploratorio del Corpus (EDA)

El EDA no es un paso opcional — es la base sobre la que tomamos **decisiones de diseño concretas**:

| Pregunta | Impacto en el diseño |
|---|---|
| ¿Qué longitud tienen los artículos? | Determina el `chunk_size` óptimo |
| ¿Está balanceado el corpus entre WWI y WWII? | Informa si el retriever favorecerá una guerra |
| ¿Qué conceptos dominan? | Valida que el corpus es temáticamente relevante |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")

# ── EDA 1: Distribución de longitud de artículos ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de palabras
axes[0].hist(
    df["word_count"], bins=50, color="steelblue", edgecolor="white", alpha=0.85
)
axes[0].axvline(
    df["word_count"].mean(),
    color="red",
    linestyle="--",
    label=f'Media: {df["word_count"].mean():.0f}',
)
axes[0].axvline(
    df["word_count"].median(),
    color="orange",
    linestyle="--",
    label=f'Mediana: {df["word_count"].median():.0f}',
)
axes[0].set_title("Distribución de palabras por artículo")
axes[0].set_xlabel("Número de palabras")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

# Boxplot de caracteres
axes[1].boxplot(
    df["char_count"],
    vert=True,
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", alpha=0.6),
)
axes[1].set_title("Distribución de caracteres por artículo (boxplot)")
axes[1].set_ylabel("Número de caracteres")
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

# Estadísticas descriptivas clave
print("=== Estadísticas de longitud (palabras) ===")
print(f'  Media:       {df["word_count"].mean():.0f} palabras')
print(f'  Mediana:     {df["word_count"].median():.0f} palabras')
print(f'  Percentil 25: {df["word_count"].quantile(0.25):.0f} palabras')
print(f'  Percentil 75: {df["word_count"].quantile(0.75):.0f} palabras')
print(f'  Percentil 95: {df["word_count"].quantile(0.95):.0f} palabras')
print(f'  Máximo:      {df["word_count"].max():.0f} palabras')

### Interpretación — Distribución de longitudes

La distribución es **asimétrica a la derecha**: la mayoría de artículos son cortos-medianos, pero existen artículos muy extensos (biografías, batallas principales) que sesgan la media hacia arriba.

**Implicación para el chunking:**
- Artículos cortos (< 300 palabras): un `chunk_size` muy pequeño (256 chars) puede partir oraciones sin sentido.
- Artículos largos (> 2000 palabras): un `chunk_size` muy grande (1024 chars) reduce la granularidad del retriever y mezcla información de distintos subtemas en el mismo chunk.
- **Hipótesis inicial:** `chunk_size ≈ 512 chars` captura 1-3 párrafos coherentes para la mayoría del corpus. Lo validaremos en la sección de sensibilidad.

In [ ]:
# ── EDA 2: Distribución por sub-tema (WWI, WWII, General) ────────────────

WWI_KEYWORDS = [
    "World War I",
    "Western Front",
    "Eastern Front",
    "Trench",
    "Armistice",
    "Verdun",
    "Somme",
    "Gallipoli",
    "Austro-Hungarian",
    "Ottoman",
]
WWII_KEYWORDS = [
    "World War II",
    "Nazi",
    "Holocaust",
    "D-Day",
    "Normandy",
    "Blitzkrieg",
    "Wehrmacht",
    "Luftwaffe",
    "Pearl Harbor",
    "Hiroshima",
    "Stalingrad",
    "El Alamein",
]


def classify_article(title):
    title_lower = title.lower()
    is_ww1 = any(kw.lower() in title_lower for kw in WWI_KEYWORDS)
    is_ww2 = any(kw.lower() in title_lower for kw in WWII_KEYWORDS)
    if is_ww1 and is_ww2:
        return "Ambas/General"
    if is_ww1:
        return "WWI"
    if is_ww2:
        return "WWII"
    return "General"


df["theme"] = df["title"].apply(classify_article)
theme_counts = df["theme"].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
colors = ["#4472C4", "#ED7D31", "#A9D18E", "#FF0000"]
wedges, texts, autotexts = ax.pie(
    theme_counts.values,
    labels=theme_counts.index,
    autopct="%1.1f%%",
    colors=colors[: len(theme_counts)],
    startangle=140,
    textprops={"fontsize": 13},
)
ax.set_title("Distribución del corpus por sub-tema", fontsize=15, pad=20)
plt.tight_layout()
plt.show()

print("Conteo por sub-tema:")
print(theme_counts.to_string())

In [ ]:
from wordcloud import WordCloud, STOPWORDS

# ── EDA 3: WordCloud de títulos ───────────────────────────────────────────
# Revela qué eventos, lugares y organizaciones dominan el corpus a nivel de título

stopwords_titles = STOPWORDS | {"of", "the", "in", "at", "on", "and", "Battle", "War"}
title_text = " ".join(df["title"].tolist())

wc_titles = WordCloud(
    width=900,
    height=450,
    background_color="white",
    stopwords=stopwords_titles,
    colormap="Blues",
    max_words=100,
    collocations=False,
).generate(title_text)

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(wc_titles, interpolation="bilinear")
ax.axis("off")
ax.set_title("WordCloud — Términos más frecuentes en títulos de artículos", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import nltk

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_stopwords

# ── EDA 4: WordCloud del contenido ────────────────────────────────────────
# Tomamos las primeras 200 palabras de cada artículo para acelerar el proceso
# Usamos stopwords de NLTK para filtrar términos vacíos

en_stopwords = set(nltk_stopwords.words("english")) | STOPWORDS

# Concatenamos extractos de todos los artículos
content_sample = " ".join(
    " ".join(text.split()[:200])  # primeras 200 palabras de cada artículo
    for text in df["text"].tolist()
)

wc_content = WordCloud(
    width=900,
    height=450,
    background_color="#1a1a2e",
    stopwords=en_stopwords,
    colormap="YlOrRd",
    max_words=120,
    collocations=False,
).generate(content_sample)

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(wc_content, interpolation="bilinear")
ax.axis("off")
ax.set_title("WordCloud — Conceptos dominantes en el contenido del corpus", fontsize=14)
plt.tight_layout()
plt.show()

### Conclusiones del EDA — Decisiones de diseño

| Observación | Decisión de diseño |
|---|---|
| Artículos predominantemente sobre WWII (~60%) | El retriever responderá mejor preguntas de WWII; advertiremos esta limitación en conclusiones |
| Media de longitud ≈ 600-1000 palabras/artículo (≈ 3500-6000 chars) | `chunk_size=512` captura ~1 párrafo; `chunk_size=1024` captura 2-3 párrafos |
| WordCloud de contenido muestra: *German, British, French, forces, attack, troops* | El corpus es rico en terminología bélica → el embedding capturará semántica correcta |
| WordCloud de títulos muestra gran diversidad de batallas | Buen corpus para preguntas específicas del tipo "Battle of X" |

**Hipótesis de chunk_size:** Empezaremos evaluando tres configuraciones (256, 512, 1024) en la siguiente sección para confirmar cuál produce la mejor recuperación.

---
## 3. Preprocesamiento y Análisis de Sensibilidad al Chunking

### ¿Por qué chunking?

Los modelos de embedding tienen una ventana de tokens limitada (tipicamente 512 tokens para `e5-large`). Los artículos de Wikipedia suelen superar ese límite, por lo que **debemos dividirlos en fragmentos (chunks)** antes de indexarlos.

### Parámetros clave

- **`chunk_size`**: número máximo de caracteres por fragmento. Fragmentos pequeños → mayor granularidad, menor contexto. Fragmentos grandes → más contexto, menor especificidad.
- **`chunk_overlap`**: solapamiento entre fragmentos consecutivos. Evita cortar frases a la mitad y preserva continuidad temática.

### `RecursiveCharacterTextSplitter`

LangChain ofrece este splitter que divide primero por párrafos (`\n\n`), luego por líneas (`\n`), y finalmente por espacios. Es el más robusto para texto narrativo como Wikipedia.

In [ ]:
from langchain_core.documents import Document

# Convertimos cada artículo a un objeto Document de LangChain
# Los metadatos (título, URL) se conservarán en cada chunk para citar la fuente
docs = [
    Document(page_content=a["text"], metadata={"title": a["title"], "url": a["url"]})
    for a in articles
]

print(f"Documentos creados: {len(docs)}")
print(f"Ejemplo de metadata: {docs[0].metadata}")
print(f"Longitud del primer documento: {len(docs[0].page_content)} chars")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Prueba de Sensibilidad al Chunking ───────────────────────────────────
# Definimos 3 configuraciones que abarcan el espacio de decisión relevante
CHUNK_CONFIGS = {
    "small": {"chunk_size": 256, "chunk_overlap": 32},  # ~1-2 oraciones
    "medium": {"chunk_size": 512, "chunk_overlap": 64},  # ~1 párrafo
    "large": {"chunk_size": 1024, "chunk_overlap": 128},  # ~2-3 párrafos
}

split_results = {}
for name, cfg in CHUNK_CONFIGS.items():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"],
        chunk_overlap=cfg["chunk_overlap"],
        # Separadores: párrafos → líneas → espacios → caracteres
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    lengths = [len(c.page_content) for c in chunks]
    split_results[name] = {
        "chunks": chunks,
        "lengths": lengths,
        "n": len(chunks),
        "avg": int(sum(lengths) / len(lengths)),
        "median": int(sorted(lengths)[len(lengths) // 2]),
        "min": min(lengths),
        "max": max(lengths),
    }

# Tabla resumen
print(
    f"{'Config':<10} {'N° chunks':>10} {'Avg (chars)':>12} {'Median':>8} {'Min':>6} {'Max':>8}"
)
print("-" * 58)
for name, r in split_results.items():
    print(
        f'{name:<10} {r["n"]:>10,} {r["avg"]:>12,} {r["median"]:>8,} {r["min"]:>6,} {r["max"]:>8,}'
    )

In [ ]:
# ── Visualización de las distribuciones de longitud por configuración ─────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
palette = {"small": "#4472C4", "medium": "#ED7D31", "large": "#70AD47"}

for ax, (name, r) in zip(axes, split_results.items()):
    ax.hist(r["lengths"], bins=40, color=palette[name], edgecolor="white", alpha=0.85)
    ax.axvline(
        r["avg"], color="red", linestyle="--", linewidth=1.5, label=f'Media: {r["avg"]}'
    )
    ax.axvline(
        r["median"],
        color="black",
        linestyle=":",
        linewidth=1.5,
        label=f'Mediana: {r["median"]}',
    )
    ax.set_title(
        f'chunk_size={CHUNK_CONFIGS[name]["chunk_size"]} ({name})\nN={r["n"]:,} chunks',
        fontsize=12,
    )
    ax.set_xlabel("Longitud del chunk (chars)")
    ax.set_ylabel("Frecuencia")
    ax.legend(fontsize=9)

plt.suptitle(
    "Distribución de longitud de chunks por configuración", fontsize=14, y=1.02
)
plt.tight_layout()
plt.show()

### Decisión de Chunking

Basados en el EDA y el análisis de sensibilidad:

| Config | Pros | Contras |
|---|---|---|
| **small** (256/32) | Muy granular, preciso para hechos concretos | Pierde contexto narrativo, oraciones cortadas |
| **medium** (512/64) | Captura 1 párrafo completo, balance granularidad/contexto | — |
| **large** (1024/128) | Máximo contexto por chunk | Puede mezclar subtemas distintos en un chunk |

**Elegimos `medium` (512 chars / 64 overlap)** como configuración principal porque:
1. Los artículos de Wikipedia estructuran su información en párrafos de ~400-600 chars.
2. El modelo `multilingual-e5-large` tiene ventana de 514 tokens — 512 chars ≈ 100-130 tokens, bien dentro del límite.
3. El overlap de 64 chars (~12-15 tokens) garantiza continuidad entre fragmentos consecutivos.

> En el chatbot final podrá observarse cualitativamente si el retriever recupera chunks coherentes.

In [ ]:
# Seleccionamos la configuración medium como la principal para el RAG
chunks = split_results["medium"]["chunks"]
print(f"Chunks seleccionados (medium): {len(chunks):,}")
print(f"Ejemplo de chunk:")
print(f"  Metadata: {chunks[0].metadata}")
print(f"  Contenido (primeros 300 chars): {chunks[0].page_content[:300]}...")

---
## 4. Configuración del LLM: Mistral 7B Instruct

### ¿Por qué Mistral 7B y no llama3.2:3b?

| Característica | llama3.2:3b | **mistral:7b-instruct** |
|---|---|---|
| Parámetros | 3B | **7B** |
| Ventana de contexto | 128K | **32K** |
| Benchmark MMLU | ~58% | **~64%** |
| Instruction-following | Bueno | **Muy bueno** |
| Uso de VRAM (cuantizado) | ~2 GB | **~4.1 GB** |

Mistral 7B Instruct v0.3 tiene mejor razonamiento y síntesis de texto, lo que se traduce en respuestas históricas más precisas y coherentes. Cabe dentro de la GPU T4 de Colab (16 GB VRAM) con cuantización Q4.

> **Nota:** Necesitamos que `ollama serve` esté corriendo antes de ejecutar las siguientes celdas. Si estás en Colab, usa la terminal integrada (xterm) para ejecutar `ollama serve &`.

In [ ]:
# Instalamos dependencias del sistema y Ollama si no está disponible
!sudo apt install zstd -y -q
!if ! type ollama > /dev/null 2>&1; then \
    echo 'Instalando Ollama...' && curl -fsSL https://ollama.com/install.sh | sh; \
else \
    echo 'Ollama ya está instalado.'; \
fi

In [ ]:
# Cargamos la extensión de terminal para Colab
# En la terminal que se abre, ejecuta: ollama serve &
# Esto lanza el servidor de Ollama en segundo plano
%load_ext colabxterm
%xterm

In [ ]:
# Descargamos el modelo Mistral 7B Instruct cuantizado (Q4 por defecto en Ollama)
# El modelo pesa ~4.1 GB — puede tardar 2-5 minutos en Colab con buena conexión
!ollama pull mistral:7b-instruct

In [ ]:
from langchain_ollama import ChatOllama

# temperature=0.1: respuestas casi deterministas, importantes para precisión histórica
# Un temperature alto generaría respuestas más 'creativas' pero menos fiables en RAG
llm = ChatOllama(model="mistral:7b-instruct", temperature=0.1)

# Prueba de sanity: verificamos que el LLM responde antes de continuar
response = llm.invoke("Who started World War I and what were the main causes?")
print("=== Respuesta del LLM (sin contexto RAG) ===")
print(response.content)

---
## 5. Creación del Vector Store con FAISS

### Pipeline de indexación

```
chunks (texto) → Embedding Model → vectores float32 → FAISS Index
```

1. **`intfloat/multilingual-e5-large`**: modelo de embedding de 560M parámetros, entrenado en 94 idiomas. Produce vectores de 1024 dimensiones. A pesar de ser multilingüe, mantiene excelente rendimiento en inglés puro (MTEB benchmark).

2. **FAISS** (Facebook AI Similarity Search): librería especializada en búsqueda de vecinos más cercanos en espacios vectoriales de alta dimensión. Usa índices aproximados para escalar a millones de vectores manteniendo baja latencia.

3. **Guardado del índice**: el índice FAISS se serializa a disco. En ejecuciones posteriores se carga directamente, evitando recalcular todos los embeddings (~30 min en T4 para 3,000 artículos).

In [ ]:
import os
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Inicializamos el modelo de embeddings con normalización L2 (recomendado para e5)
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    encode_kwargs={
        "normalize_embeddings": True
    },  # necesario para similitud coseno correcta
)

index_path = "./faiss_ww_index"

if os.path.exists(index_path):
    # Cargamos el índice existente (evita re-embeddear todo el corpus)
    print("Cargando índice FAISS existente...")
    vectorstore = FAISS.load_local(
        index_path, embeddings, allow_dangerous_deserialization=True
    )
else:
    # Construimos el índice desde los chunks — puede tardar 20-40 min en T4
    print(f"Construyendo índice FAISS para {len(chunks):,} chunks...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(index_path)
    print("Índice guardado en disco.")

# El retriever buscará los 5 chunks más similares a cada consulta (k=5)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print(f"Retriever listo. Índice contiene {vectorstore.index.ntotal:,} vectores.")

---
## 6. Pipeline RAG: Pregunta-Respuesta Simple

### Arquitectura de la cadena

```
pregunta → retriever → [chunk_1, ..., chunk_5]
                              │
                    create_stuff_documents_chain
                    (concatena chunks en el prompt)
                              │
                           LLM
                              │
                          respuesta + fuentes
```

Usamos `create_retrieval_chain` (forma moderna, no deprecada) que devuelve automáticamente el contexto recuperado como lista de `Document` objects, facilitando la cita de fuentes.

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

# Prompt en inglés para alinearlo con el corpus
# Instruimos al LLM a citar fuentes y admitir incertidumbre
prompt = PromptTemplate.from_template(
    """\
You are a knowledgeable historian specializing in World War I and World War II.
Use the following context fragments retrieved from Wikipedia to answer the question.
If the answer is not in the context, say so honestly — do not hallucinate.
Always include citations to the source articles at the end of your response.

Context:
{context}

Question: {input}
Answer:"""
)

# create_stuff_documents_chain concatena todos los chunks en el campo {context}
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
# create_retrieval_chain une el retriever con la cadena de documentos
qa_chain = create_retrieval_chain(retriever, combine_docs_chain)


def format_answer(result):
    """Formatea la respuesta incluyendo título y URL de cada fuente recuperada."""
    answer = result["answer"] + "\n\n**Fuentes:**\n"
    seen = set()
    for i, doc in enumerate(result["context"], 1):
        title = doc.metadata.get("title", "Desconocido")
        url = doc.metadata.get("url", "")
        key = title  # deduplicamos por título
        if key not in seen:
            answer += f"  [{i}] {title} — {url}\n"
            seen.add(key)
    return answer


print("Cadena RAG configurada correctamente.")

In [ ]:
# ── Pruebas con preguntas históricas ─────────────────────────────────────
test_questions = [
    "What were the main causes of World War I?",
    "Describe the D-Day invasion and its significance.",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {q}")
    print("=" * 60)
    result = qa_chain.invoke({"input": q})
    print(format_answer(result))

### Observaciones sobre la calidad del RAG simple

- **Precisión de retrieval**: Las fuentes recuperadas deben corresponder temáticamente a la pregunta. Si aparecen artículos irrelevantes, podría indicar que `k=5` es demasiado alto o que el corpus tiene ruido.
- **Calidad de síntesis**: Mistral 7B Instruct organiza bien la información de múltiples fragmentos en una respuesta coherente.
- **Limitación del RAG simple**: cada pregunta es independiente — si haces preguntas de seguimiento ('¿Y el general que lo lideró?'), el modelo no tiene memoria de la pregunta anterior. Esto lo solucionamos en la siguiente sección.

---
## 7. Cadena Conversacional con Historial

### El problema del RAG sin memoria

En el QA simple, cada invocación es independiente. Si el usuario pregunta:
1. *'Tell me about the Battle of Stalingrad'*
2. *'Who commanded the German forces there?'*

La segunda pregunta ('there') pierde su referente sin historial. El retriever no sabrá buscar sobre Stalingrad.

### Solución: `create_history_aware_retriever`

```
historial + nueva pregunta
          │
          ▼
  LLM reformula la pregunta
  ('Who commanded at Stalingrad?')
          │
          ▼
      retriever
          │
          ▼
    LLM genera respuesta
```

El LLM primero **condensa** el historial + la nueva pregunta en una pregunta autónoma y explícita, que luego se envía al retriever.

In [ ]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

# Prompt para condensar el historial + nueva pregunta en una pregunta autónoma
condense_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given the conversation history and the latest user question, "
            "reformulate the question into a standalone question that can be understood "
            "without the conversation context. Keep the original language. "
            "Do NOT answer the question — only reformulate it if needed.",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)

# El retriever ahora recibe la pregunta condensada
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, condense_prompt
)

# Prompt principal del QA conversacional
qa_system_prompt = (
    "You are a knowledgeable historian specializing in World War I and World War II. "
    "Use the following retrieved context to answer the question. "
    "If you cannot find the answer in the context, say so. "
    "Be concise but informative.\n\n{context}"
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)

# Cadena final: retriever consciente del historial + generación
qa_chain_conv = create_stuff_documents_chain(llm, qa_prompt)
convo_qa_chain = create_retrieval_chain(history_aware_retriever, qa_chain_conv)

print("Cadena conversacional configurada.")

In [ ]:
# ── Demostración de conversación multi-turno ──────────────────────────────
# Mostramos cómo el historial permite preguntas de seguimiento con referencias implícitas

chat_history = []

conversation = [
    "Tell me about the Battle of Stalingrad.",
    "Who commanded the German forces there?",  # 'there' = Stalingrad
    "What happened to him after the battle?",  # 'him' = von Paulus
]

for question in conversation:
    print(f"\n{'─'*60}")
    print(f"Usuario: {question}")
    print("─" * 60)

    response = convo_qa_chain.invoke(
        {
            "input": question,
            "chat_history": chat_history,
        }
    )

    # Actualizamos el historial para la siguiente vuelta
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response["answer"]))

    print(f'Asistente: {response["answer"]}')
    # Mostramos las fuentes recuperadas
    sources = list({d.metadata["title"] for d in response["context"]})
    print(f"[Fuentes: {', '.join(sources[:3])}]")

---
## 8. Interfaz de Usuario — ChatBot Gradio

Construimos la interfaz conversacional con **Gradio Blocks**, que ofrece más flexibilidad que `gr.ChatInterface`.

### Gestión dual de historiales

| Historial | Tipo | Propósito |
|---|---|---|
| `lc_history` | `List[HumanMessage \| AIMessage]` | Pasado a LangChain en cada invocación |
| `chat_history` (Gradio) | `List[Tuple[str, str]]` | Renderizado visual en el componente `gr.Chatbot` |

Ambos historiales se resetean al presionar el botón **Limpiar**, garantizando que una nueva conversación no herede contexto de la anterior.

In [ ]:
import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage

# Historial de LangChain (estructuras internas)
lc_history = []


def respond(question, chat_history):
    """Procesa la pregunta del usuario y actualiza ambos historiales."""
    if not question.strip():
        return "", chat_history

    # Invocamos la cadena conversacional con el historial de LangChain
    reply = convo_qa_chain.invoke({"input": question, "chat_history": lc_history})

    # Actualizamos el historial interno de LangChain
    lc_history.append(HumanMessage(content=question))
    lc_history.append(AIMessage(content=reply["answer"]))

    # Formateamos respuesta con fuentes para la UI
    answer = format_answer(reply)

    # Actualizamos el historial visual de Gradio
    chat_history.append((question, answer))
    return "", chat_history


def reset_chat():
    """Limpia ambos historiales para iniciar una conversación nueva."""
    lc_history.clear()  # vaciamos el historial de LangChain
    return [], ""  # vaciamos la UI de Gradio


# Construimos la interfaz
with gr.Blocks(title="WW Historian RAG", theme=gr.themes.Soft()) as gr_blocks:
    gr.Markdown(
        """
    # Historiador de las Guerras Mundiales — RAG
    Pregunta sobre la **Primera** o **Segunda Guerra Mundial**.
    Las respuestas están fundamentadas en artículos de Wikipedia y el modelo **Mistral 7B Instruct**.
    """
    )

    chatbot = gr.Chatbot(label="Conversación", height=450)
    msg = gr.Textbox(
        label="Tu pregunta",
        placeholder="Ej: What were the main battles of World War II?",
        lines=2,
    )

    with gr.Row():
        submit_btn = gr.Button("Enviar", variant="primary")
        clear_btn = gr.Button("Limpiar conversación")

    # Enviamos con Enter o con el botón
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    submit_btn.click(respond, [msg, chatbot], [msg, chatbot])
    clear_btn.click(reset_chat, None, [chatbot, msg], queue=False)

gr_blocks.launch(inline=False, share=IN_COLAB)

In [ ]:
# Cerramos la interfaz de Gradio al finalizar el taller
gr_blocks.close()

---
## 9. Conclusiones

### Resumen del taller

Implementamos un **sistema RAG completo** especializado en la historia de las Guerras Mundiales, recorriendo todo el pipeline:

1. **Curaduría del corpus**: Filtrado semántico de Wikipedia para obtener un corpus temáticamente relevante.
2. **EDA informada**: El análisis de la distribución de longitudes y la composición temática del corpus fundamentó directamente las decisiones de chunking.
3. **Chunking con análisis de sensibilidad**: Comparamos tres configuraciones y justificamos `chunk_size=512` como balance óptimo entre granularidad y contexto.
4. **Indexación vectorial con FAISS**: Usamos embeddings multilinguales de alta calidad y búsqueda por similitud coseno.
5. **Cadena conversacional**: Incorporamos historial de conversación para preguntas de seguimiento naturales.
6. **Interfaz Gradio**: Desplegamos el chatbot con gestión correcta de los dos historiales (LangChain + Gradio).

### Tabla comparativa: configuraciones de chunking

| Config | N° chunks | Granularidad | Contexto por chunk | Recomendado para |
|---|---|---|---|---|
| small (256/32) | Mayor | Alta | Bajo | Preguntas de hechos muy concretos |
| **medium (512/64)** | **Medio** | **Media** | **Medio** | **Uso general (elegido)** |
| large (1024/128) | Menor | Baja | Alto | Preguntas de análisis y síntesis |

### Análisis crítico — Limitaciones

- **Desbalance del corpus**: el corpus tiene más artículos de WWII que de WWI debido a los keywords de filtrado; esto puede afectar la calidad del retrieval para preguntas específicas de WWI.
- **Latencia**: Mistral 7B con Ollama en Colab T4 tarda ~5-15 segundos por respuesta. En producción se usaría un servicio de inferencia optimizado (vLLM, TGI).
- **Evaluación cualitativa**: no implementamos métricas automáticas de evaluación RAG (ej. RAGAS, que mide faithfulness, answer relevancy, context precision). Esto es una mejora directa para el siguiente paso.
- **Chunking fijo**: usamos `RecursiveCharacterTextSplitter` con un tamaño fijo. Enfoques más avanzados como *semantic chunking* (dividir por cambios de tema detectados con embeddings) podrían mejorar la coherencia de cada chunk.

### Conexión con la teoría del curso

- **Transformers y embeddings** (Unidad 4): el modelo `e5-large` es un Transformer encoder fine-tuneado con contrastive learning para producir representaciones semánticas densas. La similitud coseno en el espacio de embeddings implementa la noción de *semantic similarity* vista en clase.
- **Generación de texto** (Unidad 5): Mistral 7B Instruct es un Transformer decoder con instruction-tuning, similar al GPT estudiado, pero con mejoras como Grouped Query Attention (GQA) y Sliding Window Attention (SWA).
- **RAG vs. fine-tuning**: RAG actualiza el conocimiento del sistema sin reentrenar; el fine-tuning adapta los pesos del modelo. Para dominios que cambian frecuentemente (noticias, documentos legales), RAG es la arquitectura más práctica.

### Posibles mejoras

- **Evaluación automática**: integrar [RAGAS](https://github.com/explodinggradients/ragas) para medir faithfulness y relevancy de las respuestas.
- **Búsqueda híbrida**: combinar similitud vectorial (FAISS) con BM25 (keyword matching) usando `EnsembleRetriever` de LangChain.
- **Re-ranking**: aplicar un modelo cross-encoder (ej. `ms-marco-MiniLM`) para reordenar los chunks recuperados antes de pasarlos al LLM.
- **Semantic chunking**: reemplazar `RecursiveCharacterTextSplitter` por `SemanticChunker` de LangChain Experimental.